In [ ]:
import pandas as pd

df = pd.read_csv('HR-Employee-Attrition.csv')
df.head(5)

In [ ]:
df.info()

In [ ]:
# Đếm tổng số lượng giá trị bị thiếu (Null/NaN) trong từng cột
print("--- SỐ LƯỢNG GIÁ TRỊ THIẾU TRONG TỪNG CỘT ---")
print(df.isnull().sum())

In [ ]:
# Đếm tổng số dòng bị trùng lặp hoàn toàn
so_dong_trung = df.duplicated().sum()
print(f"--- KIỂM TRA TRÙNG LẶP ---")
print(f"Tổng số dòng bị trùng lặp là: {so_dong_trung} dòng")

# Nếu có dòng trùng lặp, in thử các dòng đó ra để xem chi tiết
if so_dong_trung > 0:
    display(df[df.duplicated(keep=False)])

In [ ]:
# 1. Loại bỏ các cột không có giá trị phân tích
cols_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=cols_to_drop)

# 2. Chuyển Attrition (Yes/No) thành (1/0) để dễ tính tương quan
df['Attrition_numeric'] = df['Attrition'].map({'Yes': 1, 'No': 0})

print("Dữ liệu sau khi làm sạch còn {} cột.".format(df.shape[1]))

In [ ]:
# PHÂN TÍCH THỐNG KÊ MÔ TẢ
# Tính trung bình, min, max của Age, MonthlyIncome, YearsAtCompany

# Chọn ra 3 cột cần phân tích theo yêu cầu
cols_to_describe = ['Age', 'MonthlyIncome', 'YearsAtCompany']

# Dùng hàm describe() để lấy các thông số thống kê, sau đó làm tròn 2 chữ số thập phân
desc_stats = df[cols_to_describe].describe().round(2)

# Lọc lấy đúng 3 thông số: Trung bình (mean), Nhỏ nhất (min) và Lớn nhất (max)
summary_table = desc_stats.loc[['mean', 'min', 'max']]

print("BẢNG THỐNG KÊ MÔ TẢ (Trung bình, Min, Max):")
display(summary_table)

# In thêm vài dòng Insight để bỏ vào báo cáo Word
print("\n--- INSIGHT NHANH TỪ THỐNG KÊ MÔ TẢ ---")
print(f"- Tuổi trung bình của nhân viên là: {summary_table.loc['mean', 'Age']} tuổi (Nhỏ nhất: {summary_table.loc['min', 'Age']}, Lớn nhất: {summary_table.loc['max', 'Age']})")
print(f"- Mức lương trung bình là: {summary_table.loc['mean', 'MonthlyIncome']}$ (Thấp nhất: {summary_table.loc['min', 'MonthlyIncome']}$, Cao nhất: {summary_table.loc['max', 'MonthlyIncome']}$)")
print(f"- Thời gian gắn bó trung bình là: {summary_table.loc['mean', 'YearsAtCompany']} năm (Lâu nhất lên tới {summary_table.loc['max', 'YearsAtCompany']} năm)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Danh sách các biến số quan trọng cần kiểm tra ngoại lệ và phân bố
cols_to_check = ['MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany']

# Thiết lập kích thước tổng thể cho cụm 3 biểu đồ
plt.figure(figsize=(16, 5))

for i, col in enumerate(cols_to_check, 1):
    plt.subplot(1, 3, i) # Tạo ra 1 hàng, 3 cột biểu đồ
    
    # Vẽ Boxplot, trục x là tình trạng Nghỉ việc, trục y là giá trị của cột tương ứng
    sns.boxplot(x='Attrition', y=col, data=df, palette='Set2')
    
    # Định dạng tiêu đề và nhãn
    plt.title(f'Phân bố {col} theo Attrition', fontsize=13, weight='bold')
    plt.xlabel('Nghỉ việc (Attrition)', fontsize=11)
    plt.ylabel(col, fontsize=11)

# Tự động căn chỉnh khoảng cách giữa các biểu đồ cho đẹp mắt
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Attrition', data=df, palette='Set2')
plt.title('Tỷ lệ nhân viên nghỉ việc (Attrition)', fontsize=14)

# Hiển thị % trên cột
total = len(df)
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 10, f'{height/total:.1%}', ha="center")
plt.show()

In [ ]:
# 1. Tạo cột Nhóm tuổi (Age_Group)
bins = [17, 25, 35, 45, 55, 100]
labels = ['18-25', '26-35', '36-45', '46-55', '56+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)

# 2. Tạo bảng thống kê trung gian
age_stats = pd.crosstab(df['Age_Group'], df['Attrition'])

# In bảng ra màn hình để báo cáo
print("BẢNG THỐNG KÊ SỐ LƯỢNG:\n")
display(pd.crosstab(df['Age_Group'], df['Attrition'], margins=True, margins_name='Tổng cộng'))

# 3. Vẽ biểu đồ Cột chồng (Stacked Bar Chart)
# Chú ý: Cột 'No' vẽ trước (nằm dưới), cột 'Yes' vẽ sau (nằm trên)
ax = age_stats.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#66c2a5', '#fc8d62'], edgecolor='white')

# Thêm tiêu đề và định dạng trục
plt.title('Cơ cấu nhân sự và Tình trạng nghỉ việc theo Nhóm tuổi', fontsize=14, weight='bold')
plt.xlabel('Nhóm tuổi', fontsize=12)
plt.ylabel('Tổng số lượng nhân viên', fontsize=12)
plt.xticks(rotation=0) # Giữ cho nhãn trục X nằm ngang dễ đọc
plt.legend(title='Nghỉ việc (Attrition)')

# Vòng lặp để ghi số liệu trực tiếp vào LÕI của từng phần màu trên cột
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy() 
    if height > 0: # Chỉ in số nếu nhóm đó có người
        ax.text(x + width/2, 
                y + height/2, 
                f'{int(height)}', 
                horizontalalignment='center', 
                verticalalignment='center',
                fontsize=11, color='black', weight='bold')

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df, palette='Set2')
plt.title('Mức lương hàng tháng theo tình trạng nghỉ việc', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(y='JobRole', hue='Attrition', data=df, palette='Set2', order=df['JobRole'].value_counts().index)
plt.title('Số lượng nghỉ việc theo Vị trí công việc', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='OverTime', hue='Attrition', data=df, palette='Set2')
plt.title('Ảnh hưởng của làm thêm giờ đến nghỉ việc', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 10))
# Chỉ chọn các biến số để tính tương quan
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
corr = df[numeric_cols].corr()

# Vẽ heatmap
sns.heatmap(corr, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Ma trận tương quan giữa các biến số', fontsize=16)
plt.show()

### 📌 Tổng kết Insight từ quá trình Exploratory Data Analysis (EDA)

Qua quá trình trực quan hóa và khám phá dữ liệu, chúng ta rút ra được các điểm đáng chú ý sau để phục vụ cho việc xây dựng Dashboard và tinh chỉnh Model Machine Learning:

* **Tỷ lệ nghỉ việc (Attrition Rate):** Tỷ lệ nhân viên rời đi chiếm **16.1%**. Dữ liệu đang ở trạng thái mất cân bằng (imbalanced). Cần lưu ý sử dụng các kỹ thuật như SMOTE hoặc thiết lập class weights khi huấn luyện mô hình để tránh hiện tượng thiên lệch (bias) dự đoán nghiêng về nhóm đa số (No Attrition).
* **Yếu tố Độ tuổi (Age):** Nhân viên trẻ dưới 30 tuổi có xu hướng nhảy việc cao nhất. Ngược lại, nhóm nhân viên trên 40 tuổi cho thấy sự ổn định và mức độ gắn bó lâu dài hơn với công ty.
* **Yếu tố Thu nhập (Monthly Income):** Nhóm nhân viên có mức lương dưới **5,000 USD/tháng** đối mặt với nguy cơ nghỉ việc cao nhất. Nhìn chung, thu nhập trung bình của nhóm nhân viên đã nghỉ việc thấp hơn rõ rệt so với nhóm tiếp tục ở lại.
* **Đặc thù Vị trí (Job Role):** Các vị trí `Laboratory Technician`, `Sales Executive`, và `Research Scientist` ghi nhận số lượng nghỉ việc nhiều nhất. Trong khi đó, nhóm nhân sự cấp quản lý (Manager, Research Director) có tỷ lệ nghỉ việc cực kỳ thấp.
* **Áp lực Làm thêm giờ (OverTime):** Trạng thái làm thêm giờ (OverTime = Yes) là một đặc trưng (feature) có sức ảnh hưởng cực lớn đến quyết định rời đi. Tỷ lệ nghỉ việc ở nhóm có làm thêm giờ cao đột biến so với nhóm không làm thêm giờ.
* **Hiện tượng Đa cộng tuyến (Multicollinearity):** Phân tích ma trận tương quan cho thấy các biến `JobLevel`, `MonthlyIncome`, và `TotalWorkingYears` có mối quan hệ tuyến tính rất mạnh với nhau (thời gian làm việc lâu -> chức vụ cao -> lương cao). Cần cân nhắc loại bỏ bớt các biến này ở bước tinh chỉnh mô hình để giảm độ phức tạp và tránh nhiễu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTENC

# Các thuật toán Học máy
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
print("⏳ Bắt đầu quá trình tiền xử lý dữ liệu...")

# 1. Tách biến mục tiêu (y) và biến độc lập (X)
y = df['Attrition_numeric']
cols_to_drop_ml = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours', 'Attrition', 'Attrition_numeric']
X = df.drop(columns=[col for col in cols_to_drop_ml if col in df.columns])

# 2. Feature Engineering (Kỹ thuật đặc trưng)
X['Tenure_Ratio'] = X['YearsAtCompany'] / X['TotalWorkingYears'].clip(lower=1)
X['Promotion_Stagnation'] = X['YearsSinceLastPromotion'] / X['YearsAtCompany'].clip(lower=1)
X['Income_Age_Ratio'] = X['MonthlyIncome'] / X['Age']

# 3. One-Hot Encoding cho các biến phân loại
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# 4. Chia tập Train/Test theo tỷ lệ 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ Đã chia tập Train ({X_train.shape[0]} dòng) và Test ({X_test.shape[0]} dòng).")

In [ ]:
# =======================================================
# BƯỚC 1: ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP DỮ LIỆU MẤT CÂN BẰNG
# =======================================================
print("⏳ Đang huấn luyện trên tập GỐC (Mất cân bằng)...")

# 1. Chuẩn hóa tập Train gốc
scaler_orig = StandardScaler()
X_train_scaled = pd.DataFrame(scaler_orig.fit_transform(X_train), columns=X_train.columns)
X_test_scaled_orig = pd.DataFrame(scaler_orig.transform(X_test), columns=X_train.columns)

# 2. Huấn luyện 4 mô hình cơ bản
log_orig = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_scaled, y_train)
rf_orig = RandomForestClassifier(random_state=42).fit(X_train_scaled, y_train)
svm_orig = SVC(kernel='linear', random_state=42).fit(X_train_scaled, y_train)
nn_orig = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42).fit(X_train_scaled, y_train)

# 3. Hàm tính Metrics (Lớp Attrition = 1)
def calculate_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'Accuracy': round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 3),
        'Recall': round(recall_score(y_true, y_pred, pos_label=1), 3),
        'F1-Score': round(f1_score(y_true, y_pred, pos_label=1), 3)
    }

# 4. Thu thập và in kết quả
metrics_orig = [
    calculate_metrics(y_test, log_orig.predict(X_test_scaled_orig), "Logistic Regression (Gốc)"),
    calculate_metrics(y_test, rf_orig.predict(X_test_scaled_orig), "Random Forest (Gốc)"),
    calculate_metrics(y_test, svm_orig.predict(X_test_scaled_orig), "SVM (Gốc)"),
    calculate_metrics(y_test, nn_orig.predict(X_test_scaled_orig), "Neural Network (Gốc)")
]

df_orig = pd.DataFrame(metrics_orig)
df_orig.index = range(1, len(df_orig) + 1)
print("BẢNG TỔNG KẾT TRÊN TẬP DỮ LIỆU GỐC (Recall rất thấp do mất cân bằng):\n")
display(df_orig)

In [ ]:
# =======================================================
# BƯỚC 2: CÂN BẰNG DỮ LIỆU (SMOTENC) VÀ HUẤN LUYỆN
# =======================================================
numeric_cols = ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction',
                'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome',
                'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating',
                'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears',
                'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
                'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager',
                'Tenure_Ratio', 'Promotion_Stagnation', 'Income_Age_Ratio']

cat_indices = [i for i, col in enumerate(X_train.columns) if col not in numeric_cols]

print("⏳ Đang chạy SMOTENC để cân bằng lớp thiểu số...")
smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)
X_train_resampled, y_train_resampled = smote_nc.fit_resample(X_train, y_train)

# Chuẩn hóa (Scaling)
scaler = StandardScaler()
X_train_resampled_scaled = pd.DataFrame(scaler.fit_transform(X_train_resampled), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_train.columns)

# Huấn luyện 4 mô hình trên tập đã cân bằng
print("⏳ Đang huấn luyện Logistic Regression...")
log_reg = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_resampled_scaled, y_train_resampled)

print("⏳ Đang dò tìm tham số Random Forest (GridSearchCV)...")
param_grid = {'n_estimators': [100, 200], 'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_resampled_scaled, y_train_resampled)
rf_best = grid_search.best_estimator_

print("⏳ Đang huấn luyện Support Vector Machine (SVM)...")
svm_model = SVC(kernel='linear', random_state=42, probability=True).fit(X_train_resampled_scaled, y_train_resampled)

print("⏳ Đang huấn luyện Neural Network (MLP)...")
nn_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42).fit(X_train_resampled_scaled, y_train_resampled)

print("✅ Hoàn tất huấn luyện 4 mô hình chính thức!")

In [ ]:
# =======================================================
# BƯỚC 3: ĐÁNH GIÁ MÔ HÌNH VÀ VẼ LƯỚI CONFUSION MATRIX
# =======================================================
# 1. Thu thập kết quả
y_preds = [
    ("Logistic Regression", log_reg.predict(X_test_scaled)),
    ("Random Forest", rf_best.predict(X_test_scaled)),
    ("Support Vector Machine", svm_model.predict(X_test_scaled)),
    ("Neural Network", nn_model.predict(X_test_scaled))
]

metrics_list = [calculate_metrics(y_test, y_pred, name) for name, y_pred in y_preds]
df_results = pd.DataFrame(metrics_list)
df_results.index = range(1, len(df_results) + 1)

print("BẢNG TỔNG KẾT SAU KHI ÁP DỤNG SMOTENC:\n")
display(df_results)

# 2. Vẽ Confusion Matrix cho cả 4 mô hình dưới dạng lưới (Grid 2x2)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, y_pred) in enumerate(y_preds):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Ở lại (0)', 'Nghỉ việc (1)'], 
                yticklabels=['Ở lại (0)', 'Nghỉ việc (1)'], ax=axes[i])
    axes[i].set_ylabel('Thực tế (Actual)', fontsize=11)
    axes[i].set_xlabel('Dự đoán (Predicted)', fontsize=11)
    axes[i].set_title(f'Confusion Matrix - {name}', fontsize=13, weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# =======================================================
# BƯỚC 4: PHÂN TÍCH NGUYÊN NHÂN VÀ ĐÓNG GÓI SẢN PHẨM
# =======================================================
# 1. Trực quan hóa Feature Importance từ Random Forest
importances = rf_best.feature_importances_
feature_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_df, palette='viridis')
plt.title('Top 10 Yếu tố ảnh hưởng đến Nghỉ việc (Random Forest)', fontsize=14, weight='bold')
plt.xlabel('Trọng số quan trọng')
plt.show()

# 2. Đóng gói mô hình và Dữ liệu
joblib.dump(log_reg, 'LogisticRegression_HR_Model.pkl') # Lưu LogReg vì điểm dự đoán tốt nhất
joblib.dump(rf_best, 'RandomForest_HR_Model.pkl')       # Lưu RF để giải thích nguyên nhân
joblib.dump(scaler, 'StandardScaler_HR.pkl')
joblib.dump(X_encoded.columns.tolist(), 'Model_Features.pkl')

df_powerbi = X.copy()
df_powerbi['Attrition'] = y.map({1: 'Yes', 0: 'No'})
output_filename = 'HR_Attrition_Advanced_for_PowerBI.csv'
df_powerbi.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"\n✅ Đã đóng gói mô hình AI và xuất file Data ({output_filename}) thành công cho Power BI!")